# Thresholding and Morphology

> **Beginner · Binary image processing**


## Why this matters

Binary images are the bridge between raw pixels and measurable objects. Threshold selection and morphological cleanup are best learned as one pipeline.

**Where it appears:** Document scanning, part inspection, foreground cleanup, table extraction, and mask post-processing.


## Learning Objectives

- Apply global, Otsu, and adaptive thresholding, and know when to use each
- Understand why a single global threshold fails under uneven lighting
- Build an automatic threshold-method selector based on image characteristics
- Apply erosion, dilation, opening, and closing correctly on binary images
- Choose structuring element shape/size deliberately, not by trial and error
- Build a noise-cleanup pipeline for binary masks using morphology


## Prerequisites

07 Filtering, Convolution, and Noise; 08 Contrast, Histograms, and Image Enhancement

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.threshold`, `cv2.adaptiveThreshold`, Otsu thresholding, `cv2.erode`, `cv2.dilate`, `cv2.morphologyEx`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Thresholding and Binarization

Thresholding converts grayscale to binary. A single **global** threshold
works only when lighting is uniform. **Otsu's method** automatically picks
the global threshold that best separates a bimodal histogram -- no manual
tuning, but still a single value for the whole image. **Adaptive**
thresholding computes a local threshold per neighborhood, correctly
handling images with a lighting gradient (e.g. a photographed page with
a shadow across it).


### Morphological Operations

Morphological operations act on binary (or grayscale) images using a
structuring element (kernel shape). **Erosion** shrinks white regions
(removes small noise, disconnects weak links); **dilation** grows them
(fills small holes, connects nearby regions). **Opening** (erode then
dilate) removes small bright noise while preserving overall shape size;
**closing** (dilate then erode) fills small dark holes. These are the
standard cleanup step after any thresholding operation.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Thresholding and Binarization


### 1. Global thresholding and its failure mode

Build a scene with a lighting gradient across it, and show a single global threshold cannot correctly binarize both the bright and dark halves at once.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def uneven_lighting_scene(size=(300, 400)) -> np.ndarray:
    """A gray 'page' with text-like blobs, under a  lighting gradient."""
    page = np.full(size, 200, dtype=np.uint8)
    rng = np.random.default_rng(3)
    for _ in range(25):
        x, y = rng.integers(20, size[1] - 20), rng.integers(20, size[0] - 20)
        w, h = rng.integers(15, 40), rng.integers(4, 10)
        cv2.rectangle(page, (x, y), (x + w, y + h), 40, -1)
    gradient = np.tile(np.linspace(1.4, 0.5, size[1]), (size[0], 1))
    return np.clip(page * gradient, 0, 255).astype(np.uint8)


page = uneven_lighting_scene()
_, global_thresh = cv2.threshold(page, 127, 255, cv2.THRESH_BINARY_INV)
show_grid(
    [("uneven lighting page", page), ("single global threshold=127", global_thresh)]
)

### 2. Otsu's method: automatic global threshold

Otsu removes the need to manually pick 127 -- but it still fails on the same uneven-lighting image because it's still a SINGLE threshold for the whole image.


In [ ]:
otsu_value, otsu_thresh = cv2.threshold(
    page, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)
print(f"Otsu automatically chose threshold = {otsu_value:.1f}")
show_grid([("Otsu global threshold", otsu_thresh)])

### 3. Adaptive thresholding: the correct fix

Adaptive thresholding computes a local threshold per pixel neighborhood, correctly handling the lighting gradient that defeated both previous approaches.


In [ ]:
adaptive = cv2.adaptiveThreshold(
    page,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    blockSize=25,
    C=10,
)
show_grid(
    [
        ("original", page),
        ("Otsu (fails under gradient)", otsu_thresh),
        ("adaptive (handles gradient)", adaptive),
    ]
)

### 4. An automatic method selector

Estimate lighting uniformity (std of a heavily blurred version of the image) and pick Otsu vs adaptive automatically -- a small but genuinely useful utility.


In [ ]:
def auto_threshold(gray: np.ndarray, uniformity_std_limit: float = 12.0) -> np.ndarray:
    """Pick Otsu for roughly-uniform lighting, adaptive thresholding otherwise."""
    illumination = cv2.GaussianBlur(gray, (51, 51), 0)
    uniformity = float(illumination.std())
    if uniformity < uniformity_std_limit:
        _, result = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        method = "otsu"
    else:
        result = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 10
        )
        method = "adaptive"
    print(f"Illumination std={uniformity:.1f} -> chose method: {method}")
    return result


_ = auto_threshold(page)

## Part 2: Morphological Operations


### 1. Erosion and dilation basics

Build a noisy binary mask (main blob + small noise specks) to make the effect of each operation visually obvious.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

# Load a real image with some texture (coins on a surface)
img = load_real_image("images/objects", "coins.jpg", cv2.IMREAD_GRAYSCALE)

# Threshold it such that coins are white.
# Due to lighting and texture, this will create a noisy mask with holes inside the coins and noise in the background.
_, mask = cv2.threshold(img, 120, 255, cv2.THRESH_BINARY_INV)

# Crop it to a smaller region to clearly see the noise and holes
mask = mask[150:450, 200:500]

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
eroded = cv2.erode(mask, kernel, iterations=1)
dilated = cv2.dilate(mask, kernel, iterations=1)

show_grid(
    [
        ("noisy real mask (thresholded)", mask),
        ("eroded (shrinks + removes dots)", eroded),
        ("dilated (grows + fills holes)", dilated),
    ]
)

### 2. Opening and closing as a cleanup pipeline

Chain opening (remove small noise) then closing (fill small holes) into a single reusable `clean_mask` function -- the standard pre-processing step before contour detection, covered in the next notebook.


In [ ]:
def clean_mask(mask: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    """Remove small noise specks and fill small holes, preserving overall blob size/shape."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return closed


cleaned = clean_mask(mask)
show_grid([("before", mask), ("after open+close", cleaned)])
print(
    f"Foreground pixels before: {int((mask > 0).sum())}, after: {int((cleaned > 0).sum())}"
)

### 3. Structuring element shape matters

Rectangular kernels bias toward preserving/creating axis-aligned structures; elliptical kernels are more isotropic (shape-neutral) and usually the right default for natural blobs.


In [ ]:
rect_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
ellipse_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))

dilated_rect = cv2.dilate(mask, rect_kernel)
dilated_ellipse = cv2.dilate(mask, ellipse_kernel)

show_grid(
    [
        ("dilate w/ RECT kernel", dilated_rect),
        ("dilate w/ ELLIPSE kernel", dilated_ellipse),
    ]
)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Thresholding and Binarization: Local Sauvola Adaptive Thresholding

Adaptive thresholding methods like Otsu fail when an image contains intense shadow gradients across the document surface. Here, we implement a local thresholding algorithm (Sauvola's method) that uses local standard deviation to handle severe shadows.


In [ ]:
# Generate text document scene with heavy shadowing
img = cv2.resize(
    load_real_image("images/objects", "sudoku.png", cv2.IMREAD_GRAYSCALE), (400, 300)
)  # Uneven lighting background
text = np.full((300, 400), 255, dtype=np.uint8)
cv2.putText(text, "SAMPLED TEXT", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 0, 3)
cv2.putText(text, "CV PIPELINE", (50, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 0, 3)
shadowed = cv2.bitwise_and(img, text)

# Calculate local mean and standard deviation using box filters
ksize = 25
mean = cv2.boxFilter(shadowed, cv2.CV_32F, (ksize, ksize))
sq_mean = cv2.boxFilter(shadowed.astype(np.float32) ** 2, cv2.CV_32F, (ksize, ksize))
std = np.sqrt(np.clip(sq_mean - mean**2, 0, None))

# Sauvola formula parameters: threshold = mean * (1 + k * (std / R - 1))
k = 0.2
R = 128.0  # Standard deviation scale factor
sauvola_thresh = mean * (1.0 + k * (std / R - 1.0))

# Perform binarization
binary = np.where(shadowed >= sauvola_thresh, 255, 0).astype(np.uint8)

print("Sauvola adaptive thresholding complete.")
show_grid(
    [
        ("Shadowed Text", shadowed),
        ("Sauvola Thresholds", sauvola_thresh.astype(np.uint8)),
        ("Binarized Result", binary),
    ]
)

### Mini Project — Morphological Operations: Extracting Table Structures

In Document Analysis systems, separating horizontal and vertical grid lines is required before performing OCR. By using morphological opening operations with custom line-shaped kernels, we can isolate horizontal and vertical structures separately.


In [ ]:
# Extracting grid lines from a real sudoku image
img = load_real_image("images/objects", "sudoku.png", cv2.IMREAD_GRAYSCALE)

# Binarize the image to isolate dark ink (text and grid lines)
# We use adaptive thresholding because of the uneven lighting shadow
mask = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 15, 10
)

# Define a horizontal structuring element (horizontal line shape)
# The length is chosen to be large enough to filter out text but keep long lines
horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
# Define a vertical structuring element (vertical line shape)
vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))

# Isolate horizontal lines
only_horizontal = cv2.morphologyEx(mask, cv2.MORPH_OPEN, horiz_kernel)

# Isolate vertical lines
only_vertical = cv2.morphologyEx(mask, cv2.MORPH_OPEN, vert_kernel)

print("Grid segmentation completed on real image.")
show_grid(
    [
        ("Binarized Sudoku", mask),
        ("Isolated Horizontal Lines", only_horizontal),
        ("Isolated Vertical Lines", only_vertical),
    ]
)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Thresholding and Binarization
1. Plot the grayscale histogram of `page` and visually locate the Otsu threshold on it.
2. Sweep adaptive thresholding's `blockSize` (11, 25, 51) and describe the trade-off in markdown.
3. Test `auto_threshold` on a uniformly-lit synthetic image and confirm it picks Otsu.

Use the empty cell below to work through them.


#### Solutions — Thresholding and Binarization

In [ ]:
# Solution 1: Plot grayscale histogram and locate Otsu threshold
def plot_otsu_threshold_loc(image: np.ndarray) -> None:
    """Find Otsu threshold and plot it along with image histogram."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    val, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    plt.figure(figsize=(6, 4))
    plt.hist(gray.ravel(), bins=256, range=[0, 256], color="gray", alpha=0.7)
    plt.axvline(val, color="r", linestyle="--", label=f"Otsu Thresh ({int(val)})")
    plt.title("Grayscale Histogram and Otsu Threshold")
    plt.xlabel("Intensity")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Solution 2: Sweep adaptive thresholding blockSize
# Explanation: The parameter `blockSize` defines the size of the pixel neighborhood used to calculate
# the local threshold. A small `blockSize` (e.g. 11) resolves high-frequency variations but is
# sensitive to text inner noise. A large `blockSize` (e.g. 51) averages out local noise but fails to
# adapt to sharp, tight lighting changes (shadow boundaries).

In [ ]:
# Solution 3: Test auto_threshold on uniformly-lit synthetic image
def test_auto_threshold() -> None:
    """Verify if auto_threshold defaults to Otsu binarization under uniform lighting."""
    img = np.zeros((100, 100), dtype=np.uint8)
    img[30:70, 30:70] = 200  # Bright central region

    val_otsu, _ = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print("Detected Otsu threshold value on uniform image:", val_otsu)


plot_otsu_threshold_loc(load_real_image("images/documents", "receipt.jpg"))
test_auto_threshold()

### Exercises — Morphological Operations
1. Write `morphological_gradient(mask)` using `cv2.MORPH_GRADIENT` and explain in markdown what it highlights.
2. Sweep kernel size 3, 5, 9, 15 through `clean_mask` and note when the main blob itself starts eroding away.
3. Use `cv2.MORPH_CROSS` and compare its effect on a diagonal line shape vs the ellipse kernel.

Use the empty cell below to work through them.


#### Solutions — Morphological Operations

In [ ]:
# Solution 1: morphological_gradient using cv2.MORPH_GRADIENT
def morphological_gradient(mask: np.ndarray) -> np.ndarray:
    """Compute morphological gradient outline of mask boundary."""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    # Morphological gradient = Dilation - Erosion
    return cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, kernel)

In [ ]:
# Solution 2: Sweep kernel size through clean_mask
# Explanation: Opening kernel size determines the scale of objects removed. A kernel size of 3
# cleans small dust noise without affecting target shapes. If kernel scale increases to 15+,
# the kernel becomes larger than the foreground objects, resulting in complete erosion of the
# target shapes.

In [ ]:
# Solution 3: Compare MORPH_CROSS vs MORPH_ELLIPSE kernels
def compare_morph_kernels() -> None:
    """Demonstrate difference of MORPH_CROSS and MORPH_ELLIPSE kernels on diagonal lines."""
    img = np.zeros((100, 100), dtype=np.uint8)
    cv2.line(img, (10, 10), (90, 90), 255, 4)  # Diagonal line

    kernel_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))
    kernel_ellipse = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    eroded_cross = cv2.erode(img, kernel_cross)
    eroded_ellipse = cv2.erode(img, kernel_ellipse)

    print("Active pixels after cross erosion:", np.sum(eroded_cross > 0))
    print("Active pixels after ellipse erosion:", np.sum(eroded_ellipse > 0))


# Test functions
img = load_real_image("images/documents", "text.png", cv2.IMREAD_GRAYSCALE)
grad = morphological_gradient(img)
compare_morph_kernels()

## Summary

You can make a binary mask, diagnose its failures, and use a structuring element to repair noise, gaps, or touching regions.

- **Best Practices:** Visualize every intermediate mask, tie kernel size to object scale, and choose adaptive methods when lighting is non-uniform.
- **Common Pitfalls:** Using one global threshold under uneven lighting, applying morphology without a goal, and choosing a kernel that erases small objects.